### ЗАДАЧА: Триаж обращений службы поддержки

Команда поддержки получает пакет строк с обращениями от разных сервисов.
Нужно обработать их так, чтобы:
- корректные обращения попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, в каких каналах остались не подтверждённые обращения,
- а также какова средняя длительность обработки по уровням приоритета.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестный уровень приоритета или канал,
а часть передаёт неправильный флаг подтверждения.
        


In [2]:
# incident_id|service|severity|duration_min|channel|acknowledged
rows = [
    'INC-100|checkout|critical|12|pager|yes',
    'INC-101|search|high|7|slack|no',
    'INC-102|billing|medium|zero|email|yes',
    'INC-103|video|critical|-3|pager|no',
    'INC-104|feed|warning|5|slack|yes',
    'INC-105|auth|low|2|sms|no',
    'INC-106|cdn|high|4|email|maybe',
    'INC-107|ml|medium|9|slack|no',
]


class IncidentProcessingError(Exception):
    pass


class IncidentFormatError(IncidentProcessingError):
    pass


class SeverityError(IncidentProcessingError):
    pass


class DurationError(IncidentProcessingError):
    pass


class ChannelError(IncidentProcessingError):
    pass


class AcknowledgedFlagError(IncidentProcessingError):
    pass


def parse_incident(row):
    # TODO: split строку по '|'
    # TODO: убрать лишние пробелы у частей через strip()
    parts = row.strip().split('|')
    # TODO: ожидать 6 частей: incident_id, service, severity, duration_raw, channel, acknowledged_raw
    if len(parts) != 6:
    # TODO: если частей не 6 -> raise IncidentFormatError(...)
        raise IncidentFormatError(f"Некорректное количество частей: {len(parts)}")
    incident_id, service, severity, duration_raw, channel, acknowledged_raw = parts
    # TODO: duration_raw преобразовать в float
    try:
        duration = float(duration_raw)
    # TODO: при ошибке преобразования использовать raise DurationError(...) from exc
    except ValueError as e:
        raise DurationError(f"Некорректное значение длительности: '{duration_raw}'") from e
    # TODO: проверить, что duration > 0
    if duration <= 0:
        raise DurationError(f"Длительность должна быть положительной: {duration}")
    # TODO: проверить severity в {'low', 'medium', 'high', 'critical'}
    valid_severities = {'low', 'medium', 'high', 'critical'}
    if severity not in valid_severities:
        raise SeverityError(f"Неизвестный уровень приоритета: '{severity}'. Допустимые: {valid_severities}")
    # TODO: проверить channel в {'email', 'slack', 'pager'}
    valid_channels = {'email', 'slack', 'pager'}
    if channel not in valid_channels:
        raise ChannelError(f"Неизвестный канал: '{channel}'. Допустимые: {valid_channels}")
    # TODO: проверить acknowledged_raw в {'yes', 'no'}
    if acknowledged_raw not in {'yes', 'no'}:
        raise AcknowledgedFlagError(f"Некорректный флаг подтверждения: '{acknowledged_raw}'. Допустимые: 'yes', 'no'")
    # TODO: превратить acknowledged_raw в bool
    acknowledged = acknowledged_raw == 'yes'
    # TODO: вернуть словарь с разобранными полями
    return {
        'incident_id': incident_id,
        'service': service,
        'severity': severity,
        'duration_min': duration,
        'channel': channel,
        'acknowledged': acknowledged
    }
    

def process_batch(rows):
    # TODO: создать списки incidents и errors
    incidents = []
    errors = []
    # TODO: пройтись по rows циклом
    for row in rows:
    # TODO: внутри try вызвать parse_incident(row)
        try:
            incident = parse_incident(row)
    # TODO: валидный incident добавить в incidents
            incidents.append(incident)
    # TODO: IncidentProcessingError сохранить в errors как (row, error_type, message)
        except IncidentProcessingError as e:
            errors.append((row, type(e).__name__, str(e)))
    # TODO: вернуть (incidents, errors)
    return incidents,errors
# TODO: вызвать process_batch(rows)
incidents, errors = process_batch(rows)
# TODO: вывести количество валидных инцидентов и количество ошибок
print(f"Валидные инциденты: {len(incidents)}")
print(f"Ошибки: {len(errors)}")
# TODO: собрать error_counts: dict[str, int]
error_counts = {}
for _, error_type, _ in errors:
    error_counts[error_type] = error_counts.get(error_type, 0) + 1
print("\nСтатистика по ошибкам:")
for error_type, count in error_counts.items():
    print(f"  {error_type}: {count}")
# TODO: собрать unacked_by_channel: dict[str, list[str]] только для acknowledged == False
from collections import defaultdict
unacked_by_channel = defaultdict(list)
for incident in incidents:
    if not incident['acknowledged']:
        unacked_by_channel[incident['channel']].append(incident['incident_id'])
print("\nНеподтверждённые инциденты по каналам:")
for channel, incident_ids in unacked_by_channel.items():
    print(f"  {channel}: {incident_ids}")
# TODO: собрать average_duration_by_severity только по валидным строкам
from collections import defaultdict
severity_durations = defaultdict(list)
for incident in incidents:
    severity_durations[incident['severity']].append(incident['duration_min'])
average_duration_by_severity = {}
for severity, durations in severity_durations.items():
    average_duration_by_severity[severity] = sum(durations) / len(durations)
print("\nСредняя длительность по уровням приоритета:")
for severity, avg_duration in average_duration_by_severity.items():
    print(f"  {severity}: {avg_duration:.2f} мин")
# TODO: найти longest_incident среди валидных инцидентов по duration_min
if incidents:
    longest_incident = max(incidents, key=lambda x: x['duration_min'])
    print(f"\nСамый длительный инцидент: {longest_incident}")
else:
    print("\nСамый длительный инцидент: не найден (нет валидных инцидентов)")
# TODO: красиво вывести получившиеся структуры
print("\nДетализация ошибок:")
for row, error_type, message in errors:
    print(f"  {error_type}: {message} | Строка: {row}")


Валидные инциденты: 3
Ошибки: 5

Статистика по ошибкам:
  DurationError: 2
  SeverityError: 1
  ChannelError: 1
  AcknowledgedFlagError: 1

Неподтверждённые инциденты по каналам:
  slack: ['INC-101', 'INC-107']

Средняя длительность по уровням приоритета:
  critical: 12.00 мин
  high: 7.00 мин
  medium: 9.00 мин

Самый длительный инцидент: {'incident_id': 'INC-100', 'service': 'checkout', 'severity': 'critical', 'duration_min': 12.0, 'channel': 'pager', 'acknowledged': True}

Детализация ошибок:
  DurationError: Некорректное значение длительности: 'zero' | Строка: INC-102|billing|medium|zero|email|yes
  DurationError: Длительность должна быть положительной: -3.0 | Строка: INC-103|video|critical|-3|pager|no
  SeverityError: Неизвестный уровень приоритета: 'warning'. Допустимые: {'medium', 'low', 'high', 'critical'} | Строка: INC-104|feed|warning|5|slack|yes
  ChannelError: Неизвестный канал: 'sms'. Допустимые: {'email', 'pager', 'slack'} | Строка: INC-105|auth|low|2|sms|no
  Acknowledge